In [ ]:
import bw2data as bd
import bw2io as bi
import bw2regional as bwr
import geopandas as gp
from pathlib import Path

In [ ]:
bi.restore_project_directory(
    "<path to ecoinvent>",
    project_name="Palm-oil",
    overwrite_existing=True,
    switch=True,
)

In [ ]:
bd.projects.set_current("Palm-oil")

In [ ]:
bd.databases

In [ ]:
data_dir = Path.cwd().absolute() / "data"
assert data_dir.is_dir()

Tell `bw2regional` about our maps.

In [ ]:
bwr.geocollections['regions'] = {
    'filepath': str(data_dir / "malaysia.gpkg"),
    'field': 'name',
}

In [ ]:
bwr.geocollections['ecoregions'] = {
    'filepath': "/etc/data/regionalisation/lcia.gpkg",
    'field': 'OBJECTID',
}

In [ ]:
bwr.geocollections['palm'] = {
    'filepath': str(data_dir / "Malaysia.tif"),
    'nodata': 0,
}

In [ ]:
df1 = gp.read_file(bwr.geocollections['regions']["filepath"])
df2 = gp.read_file(bwr.geocollections['ecoregions']["filepath"])
intersection = gp.overlay(df1, df2, keep_geom_type=False)

In [ ]:
intersection.drop(
    inplace=True, 
    columns=[
        'CF_occ_avg_reg',
        'ECO_ID',
        'ECO_NUM',
        'FCLASS_AR',
        'FCLASS_BD',
        'FCLASS_BR',
        'FCLASS_CN',
        'FCLASS_DE',
        'FCLASS_EG',
        'FCLASS_ES',
        'FCLASS_FR',
        'FCLASS_GB',
        'FCLASS_GR',
        'FCLASS_ID',
        'FCLASS_IL',
        'FCLASS_IN',
        'FCLASS_ISO',
        'FCLASS_IT',
        'FCLASS_JP',
        'FCLASS_KO',
        'FCLASS_MA',
        'FCLASS_NL',
        'FCLASS_NP',
        'FCLASS_PK',
        'FCLASS_PL',
        'FCLASS_PS',
        'FCLASS_PT',
        'FCLASS_RU',
        'FCLASS_SA',
        'FCLASS_SE',
        'FCLASS_TLC',
        'FCLASS_TR',
        'FCLASS_TW',
        'FCLASS_UA',
        'FCLASS_US',
        'FCLASS_VN',
        'OBJECTID',
        'abbrev',
        'adm0_a3',
        'adm0_label',
        'adm0_sr',
        'admin',
        'area_sqkm',
        'check_me',
        'code_hasc',
        'code_local',
        'datarank',
        'diss_me',
        'featurecla',
        'fips',
        'fips_alt',
        'gadm_level',
        'geonunit',
        'gn_a1_code',
        'gn_id',
        'gn_level',
        'gn_name',
        'gn_region',
        'gns_adm1',
        'gns_id',
        'gns_lang',
        'gns_level',
        'gns_name',
        'gns_region',
        'gu_a3',
        'hasc_maybe',
        'iso_3166_2',
        'iso_a2',
        'labelrank',
        'latitude',
        'longitude',
        'mapcolor13',
        'mapcolor9',
        'max_label',
        'min_label',
        'min_zoom',
        'name',
        'name_alt',
        'name_ar',
        'name_bn',
        'name_de',
        'name_el',
        'name_en',
        'name_es',
        'name_fa',
        'name_fr',
        'name_he',
        'name_hi',
        'name_hu',
        'name_id',
        'name_it',
        'name_ja',
        'name_ko',
        'name_len',
        'name_local',
        'name_nl',
        'name_pl',
        'name_pt',
        'name_ru',
        'name_sv',
        'name_tr',
        'name_uk',
        'name_ur',
        'name_vi',
        'name_zh',
        'name_zht',
        'ne_id',
        'note',
        'postal',
        'provnum_ne',
        'region_cod',
        'region_sub',
        'sameascity',
        'scalerank',
        'sov_a3',
        'sub_code',
        'type',
        'type_en',
        'wikidataid',
        'wikipedia',
        'woe_id',
        'woe_label',
        'woe_name'        
    ]
)

In [ ]:
intersection['id'] = list(range(1, len(intersection) + 1))

In [ ]:
intersection.to_file(str(data_dir / 'regions_ecoregions.gpkg'), driver='GPKG')

In [ ]:
bwr.geocollections["regions - ecoregions"] = {
    "filepath": str(data_dir / 'regions_ecoregions.gpkg'),
    "field": "id",
    "is intersection": True,
    "first": 'regions',
    "second": 'ecoregions',
}

In [ ]:
occupation = bd.get_node(
    database="ecoinvent-3.10.1-biosphere",
    name="Occupation, permanent crop"
)

In [ ]:
glam = bd.Method(("GLAM", "land use"))
glam.register()

bwr.import_regionalized_cfs(
    "ecoregions",
    ("GLAM", "land use"),
    {"CF_occ_avg_reg": [occupation.key]},
    nan_value=0
)